# HW 6: the finite element method, through *Between the Nodes*

Complete the problems below. Then save your notebook and submit the `.ipynb`
file on Gradescope. Make sure you **run every code cell, including the check
cells, before saving**, so that all required output is visible in your
submission.

**How this assignment is organized.** There are four problems, one for each
class session, each built on the chapters you read for that session:

| Session | Reading | Problem |
|---|---|---|
| Fri 11/6 | Cold open, chapter 1 | 1: Galerkin for the heat shield |
| Mon 11/9 | Chapter 2, through the solve | 2: Pruitt's board |
| Wed 11/11 | Rest of chapter 2, chapter 3 | 3: Mara's puzzle, and when it matters |
| Fri 11/13 | Chapter 4 | 4: Show them |

You fill in one problem per session and submit the whole notebook once. The
written parts are graded on the *argument*, not the length.

**Hints.**

- **Keep the variable names given in the docstrings and comments.** If a
  function is documented as returning `(nodes, T)`, return that pair in that
  order. The check cells depend on it, and points are deducted otherwise.
- **Written answers go in the markdown cell provided under each part.** Plain
  text is fine — you are not required to typeset anything in LaTeX, as long as
  your notation is unambiguous and a reader can follow the argument.
- Index from $0$, as always. In Problem 2, as in Lecture 18, $n$ is the
  **number of elements**: nodes $x_0, \ldots, x_n$, tents $b_0, \ldots, b_n$.
  In Problems 1 and 3, as in Lectures 16–17, $n$ is the **polynomial degree**.
- Lengths are in millimeters and temperatures in kelvin, as in the book.
- Add scratch cells while you work; delete them before submitting.

## Part 0: Supplementary code

The first cell installs the Falkenrath-Voss thermal code the characters use in
the book (`falkenrath`), the same code that made every number and plot in it.
It takes a few seconds, needs an internet connection, and only has to succeed
once on your machine. (If it fails, ask a TA: the same package is on Canvas as
a `.zip`, and `%pip install --no-deps <that file>` works offline.)

The second cell imports it and defines two helpers you'll use in Problem 2.

In [ ]:
%pip install --quiet --no-deps https://github.com/ajgoza/between-the-nodes/archive/refs/heads/main.zip

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import quad, trapezoid

import falkenrath
from falkenrath import mara, tent, exact, KESTREL_A, KESTREL_B, UNIFORM
# KESTREL_B: the thin bondline (0.4 mm).  KESTREL_A: the legacy one (4 mm).
# UNIFORM: the same 40 mm, one material throughout (no interfaces).
for s in (KESTREL_B, KESTREL_A, UNIFORM):
    print(s.name, "| interfaces (mm):", s.interfaces, "| k per layer:", s.conductivity)


def layer_nodes(stack):
    """Tent's mesh with one element per layer: a node at each face and interface (mm)."""
    return np.concatenate(([0.0], stack.interfaces, [stack.thickness]))


def load_vector(stack, nodes):
    """The right-hand side (f, b_i) for i = 0, ..., n, for tents on `nodes`.

    Parameters
    ----------
    stack : falkenrath.Stack
        Supplies the heating f(x) = stack.heating(x)
    nodes : ndarray
        The n + 1 node positions x_0 < ... < x_n, in mm

    Returns
    -------
    F : ndarray
        Shape (n + 1,). F[i] = integral of f times b_i, built element by element
    """
    F = np.zeros(len(nodes))
    for e in range(len(nodes) - 1):
        a, b = nodes[e], nodes[e + 1]
        h = b - a
        F[e] += quad(lambda s: stack.heating(s) * (b - s) / h, a, b)[0]
        F[e + 1] += quad(lambda s: stack.heating(s) * (s - a) / h, a, b)[0]
    return F

# Problem 1: Galerkin for the heat shield

Read before this problem: the cold open and chapter 1.

Through the stack, $0 < x < L$ with $L = 40$ mm, the steady temperature obeys

$$-\bigl(k(x)\,u'\bigr)' = f(x), \qquad u(0) = T_{\text{hot}}, \quad u(L) = T_{\text{cold}},$$

where the conductivity $k(x)$ is constant in each layer (tile, glue, structure)
and **jumps** at the interfaces, and $f(x)$ is the heat the tile absorbs. Across
an interface the temperature is continuous, and so is the heat flowing through,
$k\,u'$: what leaves one layer enters the next.

Mara's code, and every Galerkin method, follows the same three steps
(Lecture 18). This problem walks through them for the heat shield.

### Part a: the subspace (concept question)

The face temperatures aren't unknown, so the approximation is written as

$$u_a(x) = g(x) + \sum_{j} c_j\,b_j(x),$$

where $g$ is any known function with $g(0) = T_{\text{hot}}$ and
$g(L) = T_{\text{cold}}$, and every basis function $b_j$ of the subspace
$\mathcal{V}$ satisfies $b_j(0) = b_j(L) = 0$.

1. Show that $u_a$ meets both boundary conditions, whatever the $c_j$ are.
2. In one or two sentences: why must the $b_j$ vanish at both faces, rather than
   just being any functions?

**YOUR ANSWER HERE:**

### Part b: from the residual to the weak form (concept question)

1. Write the residual $R(x)$ of $u_a$ in the BVP.

2. The Galerkin condition makes the residual orthogonal to $\mathcal{V}$. To
   turn that into something computable, Lecture 18 integrates by parts to the
   **weak form**. Derive it here, starting from the exact solution: multiply
   $-(k\,u')' = f$ by any $v$ in $\mathcal{V}$, integrate over the stack, and
   integrate the left side by parts so that $u$ and $v$ each carry one
   derivative.

   *Careful:* $k$ jumps at the interfaces, so do it **one layer at a time** and
   add the layers up, exactly as Lecture 18 did for the bonded rod. Write out
   the boundary terms that appear (two at the faces, two at each interface).
   Say why the face terms vanish, and why the interface terms cancel in pairs.
   You should arrive at $a(u, v) = (f, v)_{L^2}$ for every $v$ in $\mathcal{V}$,
   with $a(v, w) = \int_0^L k\,v'\,w'\,dx$.

3. Galerkin asks $u_a$ to satisfy the same weak form, tested against each basis
   function: $a(u_a, b_i) = (f, b_i)_{L^2}$ for every $i$. Substitute
   $u_a = g + \sum_j c_j b_j$ and conclude that

   $$\sum_j c_j\,a(b_j, b_i) = (f, b_i)_{L^2} - a(g, b_i)
   \qquad\text{for every } i .$$

**YOUR ANSWER HERE:**

### Part c: the linear system (concept question)

1. Write the result of part b as a linear system $A\,c = b$: give the entries of
   $A$ and of the right-hand side $b$.
2. The exact temperature $u$ is unknown. Explain why every entry of $A$ and $b$
   is nonetheless something you can compute.
3. In two sentences: nothing in parts a–c said what the basis functions $b_j$
   *are*. What does that tell you about the difference between Mara's code and
   any other Galerkin method for this BVP?

**YOUR ANSWER HERE:**

# Problem 2: Pruitt's board

Read before this problem: chapter 2, through the solve that hits zero error.

Tent uses the finite element subspace of Lecture 18: functions that are
continuous, linear on each element, and zero at both faces, with the tents
(hat functions) $b_i$ as a basis. Here the mesh has one element per layer, so
for Kestrel B there are $n = 3$ elements and nodes $x_0 = 0$, $x_1 = 24$,
$x_2 = 24.4$, $x_3 = 40$ mm.

### Part a: what changes (concept question)

In two or three sentences: what is the subspace for Mara's code, what is it for
Tent, and which of the three steps of Problem 1 is different between them?

**YOUR ANSWER HERE:**

### Part b: the element, and the Kestrel B system (concept question)

1. On element $e = [x_e, x_{e+1}]$, with width $h_e$ and conductivity $k_e$,
   only $b_e$ and $b_{e+1}$ are nonzero. Find their slopes there, and show that
   the element contributes

   $$\frac{k_e}{h_e}\begin{bmatrix} 1 & -1 \\ -1 & 1 \end{bmatrix}$$

   to the rows and columns of nodes $e$ and $e+1$.

2. Now look at one row of $A$. Interior node $i$ sits between element $i-1$
   (width $h_{i-1}$, conductivity $k_{i-1}$) on its left and element $i$ (width
   $h_i$, conductivity $k_i$) on its right. Row $i$ has three nonzero entries,
   $A_{i,i-1}$, $A_{ii}$ and $A_{i,i+1}$. Write each one as a sum of integrals
   over elements, say which elements contribute to which entry, and identify
   each contribution as an entry of the element matrix from step 1. What does
   this say about how an assembly loop over the elements builds row $i$?
   (*Notation:* $i$ and $e$ are indices; $k$ is always a conductivity.)

3. For Kestrel B, write out the two equations for the interior unknowns $u_1$
   and $u_2$: say which terms involve the face temperatures and where they go.
   What do $u_1$ and $u_2$ mean physically, and what does solving this
   $2 \times 2$ system give you?

**YOUR ANSWER HERE:**

### Part c: Tent

Complete the two functions below.

`tent_matrix` assembles the full matrix $A_{ij} = a(b_j, b_i)$ for
$i, j = 0, \ldots, n$, one element at a time, using part b.

`tent_solve` builds the one-element-per-layer mesh (`layer_nodes`), assembles
$A$, gets the right-hand side from `load_vector`, moves the face terms across,
solves for the interior temperatures with `np.linalg.solve`, and returns all
$n+1$ nodal temperatures, faces included.

In [ ]:
def tent_matrix(nodes, k):
    """The Tent matrix, assembled element by element.

    Parameters
    ----------
    nodes : ndarray
        The n + 1 node positions x_0 < ... < x_n, in mm
    k : ndarray
        The n conductivities: k[e] on element e = [x_e, x_{e+1}]

    Returns
    -------
    A : ndarray
        Shape (n + 1, n + 1), with A[i, j] = a(b_j, b_i)
    """
    ### YOUR CODE HERE ###


def tent_solve(stack):
    """Tent with one element per layer.

    Parameters
    ----------
    stack : falkenrath.Stack
        Supplies interfaces, conductivity, heating, T_hot and T_cold

    Returns
    -------
    nodes : ndarray
        The n + 1 node positions, in mm
    T : ndarray
        The n + 1 nodal temperatures, in K, including T[0] = T_hot and T[-1] = T_cold
    """
    ### YOUR CODE HERE ###

In [ ]:
"""Check the functions"""
# A small mesh you can check by hand: widths 1 and 2, conductivities 2 and 1.
A = tent_matrix(np.array([0.0, 1.0, 3.0]), np.array([2.0, 1.0]))
print("A on the small mesh:\n", A)
np.testing.assert_allclose(A, [[2, -2, 0], [-2, 2.5, -0.5], [0, -0.5, 0.5]], atol=1e-12)

# Kestrel B, one element per layer: compare with the exact temperatures.
nodes, T = tent_solve(KESTREL_B)
Te = exact(KESTREL_B)
for x_, T_ in zip(nodes, T):
    print(f"{x_:5.1f} mm   Tent {T_:9.4f} K   exact {Te.at(x_):9.4f} K")
np.testing.assert_allclose(nodes, [0, 24, 24.4, 40], atol=1e-12)
np.testing.assert_allclose(T, [Te.at(x_) for x_ in nodes], atol=1e-6)
print("TESTS PASSED")

### Part d: what locality buys (concept question)

Print `tent_matrix` for a mesh with more elements (for instance
`np.linspace(0, 40, 9)` with all $k = 1$), and look at where its nonzero entries
are. In two or three sentences: why is every entry off the three central
diagonals zero, whatever the mesh, and why was Mara's matrix for a layered
stack full instead? What does that mean for the cost of a fine mesh?

In [ ]:
### YOUR CODE HERE ###
# print the Tent matrix for a larger mesh

**YOUR ANSWER HERE:**

# Problem 3: Mara's puzzle, and when it matters

Read before this problem: the rest of chapter 2, and chapter 3.

### Part a: the puzzle

Run the cell below. It repeats Joyce's laptop check on the thick legacy stack
(stack A, `KESTREL_A`), where the polynomial can see the glue from the start: for each
polynomial degree it measures the largest slope error, the integral of the
squared slope error, and the largest temperature error. Then answer: in two or
three sentences, what do the slope and the temperature each do as the degree
doubles, and why did that look like a contradiction?

In [ ]:
x = np.linspace(0, KESTREL_A.thickness, 40001)
Te = exact(KESTREL_A)
dT, L = KESTREL_A.T_hot - KESTREL_A.T_cold, KESTREL_A.thickness
degrees = [64, 128, 256, 512]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 3.2))
for n in degrees:
    m = mara.solve(KESTREL_A, degree=n)
    slope_err = dT / L * (m.flux(x) - Te.flux(x)) / KESTREL_A.k(x)      # K/mm
    T_err = np.abs(Te.temperature(x) - m.temperature(x))
    print(f"degree {n:3d}:  largest slope error {abs(slope_err).max():4.1f} K/mm"
          f"   integral of its square {trapezoid(slope_err**2, x):7.1f}"
          f"   largest T error {T_err.max():5.2f} K")
    near = (x > 26) & (x < 30)
    ax1.plot(x[near], slope_err[near], label=f"degree {n}")
    ax2.semilogy(x, T_err + 1e-16, label=f"degree {n}")
ax1.set(xlabel="depth (mm)", ylabel="slope error (K/mm)", title="near the glue/structure interface")
ax2.set(xlabel="depth (mm)", ylabel="|temperature error| (K)", title="through the whole stack")
ax1.legend(); plt.tight_layout(); plt.show()

**YOUR ANSWER HERE:**

### Part b: Joyce's argument (concept question)

Fill in each blank, and give the one-line reason asked for. Here $e = u - u_a$
is the error, $Lu = -(k\,u')'$, and $\mathcal{V}$ is Mara's subspace.

1. The residual is $f - Lu_a = Lu - Lu_a = \underline{\qquad}$, so Galerkin
   says $(\underline{\qquad},\, v) = 0$ for every $v$ in $\mathcal{V}$.
   Integrating by parts: $\int_0^L k\,e'\,v'\,dx = \underline{\qquad}$.
   *Reason the boundary terms vanish:*
   (As Joyce notes, Mara's code never forms $Lu_a$ itself: a polynomial's flux
   jumps wherever $k$ does. It only ever uses the integrated form, which is
   why that's the line that matters.)
2. For any candidate $v$ in $\mathcal{V}$, write $u - v = e + w$. Then
   $w = \underline{\qquad}$, and $w$ is in $\mathcal{V}$ because
   $\underline{\qquad}$.
3. $\int k\,(u - v)'^2\,dx = \int k\,e'^2\,dx + 2\int \underline{\qquad}\,dx + \int k\,w'^2\,dx$.
   The middle term is zero because $\underline{\qquad}$; the last is
   $\ge 0$ because $\underline{\qquad}$. So
   $\int k\,e'^2\,dx \;\underline{\quad}\; \int k\,(u - v)'^2\,dx$ for every $v$.
4. In words: Mara's answer has the smallest $\underline{\qquad}$ of any
   function in $\mathcal{V}$. Why does Gibbs make that quantity go to zero
   even though the spike's height doesn't?
5. $e(x) = \int_0^x \underline{\qquad}\,ds$, because $e(0) = \underline{\quad}$.
   Cauchy–Schwarz with $1 \cdot e'$ gives
   $|e(x)| \le \underline{\qquad}\,\bigl(\int_0^x e'^2\,ds\bigr)^{1/2}$, and so
   $|e(x)| \to 0$ at every $x$. Which step of this argument turned a statement
   about an *integral* into one about each *point*?

**YOUR ANSWER HERE:**

### Part c: when it matters (concept question)

Run the two cells below.

The first measures Mara's largest temperature error against the polynomial
degree for three stacks: one material throughout (`UNIFORM`), the thick
bondline (`KESTREL_A`) and the thin one (`KESTREL_B`).

The second refines Tent, with $m = 1, 2, 4, \ldots, 64$ elements per layer. For
stack B it measures the largest error **between** the nodes and the largest
error **at** them. Then it repeats the run on a tile whose conductivity
**rises through the tile** (`GRADED`, $k = 1 + 2x/24$ in the tile; same glue
and structure), measuring the error at the glue's hot face, a node. There's
no exact solution for the graded tile, so the reference is Tent on a very fine
mesh.

Answer in one sentence each:

1. How does Mara's error behave on each of the three stacks?
2. How does Tent's error between the nodes change each time the elements are
   halved, and what order of accuracy is that?
3. Is Tent exact at the nodes for stack B? For the graded tile?

Then, in three or four sentences: when is a global spectral method like Mara's
the right tool for this kind of problem, when is it workable, and when should
it give way to the finite element method?

In [ ]:
x = np.linspace(0, 40, 4001)
degrees = [8, 16, 32, 64, 128, 256]
plt.figure(figsize=(6, 3.6))
for stack in (UNIFORM, KESTREL_A, KESTREL_B):
    Te = exact(stack)
    errs = [np.abs(Te.temperature(x) - mara.solve(stack, degree=n).temperature(x)).max()
            for n in degrees]
    plt.loglog(degrees, errs, "o-", label=stack.name)
plt.xlabel("polynomial degree"); plt.ylabel("largest temperature error (K)")
plt.title("Mara's code"); plt.legend(fontsize=8); plt.tight_layout(); plt.show()

In [ ]:
from falkenrath import GRADED
x = np.linspace(0, 40, 40001)
Te = exact(KESTREL_B)
ref = tent.solve(GRADED, per_layer=2048)          # reference for the graded tile
ms = [1, 2, 4, 8, 16, 32, 64]
between, at_nodes, graded = [], [], []
for m in ms:
    tb = tent.solve(KESTREL_B, per_layer=m)
    between.append(np.abs(Te.temperature(x) - tb.temperature(x)).max())
    at_nodes.append(np.abs(tb.nodal - np.array([Te.at(v) for v in tb.nodes])).max())
    graded.append(abs(tent.solve(GRADED, per_layer=m).at(24.0) - ref.at(24.0)))

print(" m    stack B, between nodes   stack B, at nodes   graded, at x = 24 mm")
for i, m in enumerate(ms):
    print(f"{m:2d}    {between[i]:10.3e} K          {at_nodes[i]:8.1e} K        {graded[i]:10.3e} K")

h = 40 / (3 * np.array(ms))                       # typical element size, mm
plt.figure(figsize=(6, 3.6))
plt.loglog(h, between, "o-", label="stack B, between nodes")
plt.loglog(h, graded, "s-", label="graded tile, at a node")
plt.loglog(h, between[-1] * (h / h[-1]) ** 2, "k--", lw=1, label="slope 2")
plt.xlabel("element size (mm)"); plt.ylabel("error (K)"); plt.title("Tent")
plt.legend(fontsize=8); plt.tight_layout(); plt.show()

**YOUR ANSWER HERE:**

# Problem 4: Show them

Read before this problem: chapter 4.

Joyce tells Luis: *"if you show them in a way they can understand, they'll
respect it."* Write the document Luis needs: **a one-page memo** (about 400
words, plus at most one figure, which you may reuse or adapt from the book or
this notebook) to the Kestrel thermal analysts. They certify adhesive
temperatures every week; they are not numerical analysts. The memo must:

- say what is wrong with certifying the thin-bondline stack with the current
  code and convergence check, with the numbers;
- say what Tent does differently, and why that fixes it;
- say what isn't known yet, honestly.

It is graded on whether its audience could act on it: leading with what they
care about, correct, clear without jargon they wouldn't use, and honest about
its limits.

**YOUR ANSWER HERE:**

### Extra credit: when $k$ changes inside an element

Luis's last image: the graded tile of Problem 3(c), whose conductivity climbs
through its thickness, $k(x) = 1 + 2x/24$ for $0 \le x \le 24$ mm. Problem 3(c)
showed that Tent isn't exact at its nodes there. Build that solver yourself,
and explain why.

1. Generalize your assembly: on an element where $k$ varies, the element
   matrix is $\dfrac{1}{h_e^2}\Bigl(\int_e k\,dx\Bigr)
   \begin{bmatrix} 1 & -1 \\ -1 & 1 \end{bmatrix}$. Why?
2. Solve with $m = 1, 2, 4, 8$ elements per layer (`load_vector` works for any
   nodes), and compare the temperature at the glue's hot face, $x = 24$ mm,
   with a reference computed on a much finer mesh. Check that you match
   `tent.solve(GRADED, per_layer=m).at(24.0)`.
3. Use chapter 2's cell 7 picture to explain why the nodes are no longer exact.

In [ ]:
### YOUR CODE HERE ###

**YOUR ANSWER HERE:**